# J4-05 · Datasets et experiments : mesurer plutôt que croire · Bilan du jour

**Formation « Orchestration d'agents & Monitoring : LangGraph + Langfuse (stack 100 % locale) »**
Jour 4 · créneau 14h30–16h30 · ⏱️ 2h (dont l'exercice 4.1, 45 min)

| | |
|---|---|
| 🎯 **Objectifs** | 1. Construire un **dataset** de cas annotés, versionné avec le code · 2. Lancer une **experiment** (`dataset.run_experiment`) avec des évaluateurs · 3. Lire les erreurs **cas par cas** · 4. Comparer des runs, et choisir un modèle **chiffres à l'appui** (exercice 4.1) · 5. Composer le tableau de bord d'un responsable Ops |
| 🧱 **Fil rouge** | Les défauts observés « à l'œil » pendant trois jours deviennent des **taux** |
| ✅ **Prérequis** | J4-01 à J4-04 ; KB indexée |

## 1. Mise en route

### [Code] Le socle du fil rouge : `src/helpdesk_ops`

Le code écrit et expliqué en J1-01 (lecture de `.env`, masquage des données personnelles, client Langfuse, création du modèle) est rangé dans le paquet **`helpdesk_ops`**, installé par `uv sync`. Chaque notebook reste **autonome** : la cellule suivante suffit.

> 💡 **Le « Tips » du Formateur :** le paquet est installé en mode *éditable* : une modification dans `src/helpdesk_ops` est prise en compte après *Kernel → Restart*. Le client Langfuse est un singleton par clé publique : en cas de doute, redémarrez le kernel.

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

## 2. Dataset, experiment, run

### [Théorie] Le banc d'essai

*Analogie :* un banc d'essai automobile. Mêmes conditions, mêmes mesures, pour chaque moteur testé : on compare des chiffres, pas des impressions de conduite.

| Terme | Sens | Ici |
|---|---|---|
| **dataset** | des cas avec leur **attendu** | 20 demandes de support (`data/eval/helpdesk_eval.json`) |
| **tâche** | ce qu'on fait tourner sur chaque cas | router, appeler un outil, qualifier |
| **évaluateur** | une fonction qui note une sortie face à l'attendu | `routage_juste`, `outil_valide`, `categorie_juste`… |
| **run** | une exécution de la tâche sur tout le dataset | `routage · granite4.1:8b` |
| **évaluateur de run** | une note sur le run entier | moyennes, p95 |

Le dataset est un fichier **versionné dans Git**, relu comme du code. `synchroniser_dataset` le pousse dans Langfuse avec des identifiants stables : relancer met à jour, ne duplique pas. Huit cas sur vingt reprennent des **défauts réellement observés** pendant les jours 1 à 3 (colonne `origine`).

In [ ]:
import pandas as pd

from helpdesk_ops import evaluation as ev

pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 60)

nom_dataset = ev.synchroniser_dataset(langfuse)
dataset = langfuse.get_dataset(nom_dataset)
print(f"dataset « {nom_dataset} » : {len(dataset.items)} cas")
print(pd.DataFrame([{**c, "kb": ", ".join(c["kb"])} for c in ev.charger_cas()["cas"]])
        [["id", "question", "specialiste", "refus", "outil", "categorie", "priorite", "origine"]].fillna("").to_string(index=False))

## 3. Première experiment : le routage

> ⚠️ **`max_concurrency=1` contre un LLM local.** Par défaut, `run_experiment` lance jusqu'à **50** cas en parallèle : contre un seul Ollama, ils font la queue et chaque latence mesurée devient fausse (vu en J3-03). Un cas à la fois : des latences honnêtes.

In [ ]:
import time

MODELE = llm.model


def experience(nom: str, tache, evaluateurs: list, variante: str = "") -> object:
    debut = time.perf_counter()
    resultat = dataset.run_experiment(name=nom, run_name=f"{nom} · {MODELE}{' · ' + variante if variante else ''}",
                                      task=tache, evaluators=evaluateurs, run_evaluators=[ev.ev_moyennes],
                                      max_concurrency=1, metadata={"modele": MODELE, "variante": variante})
    print(f"⏱ {time.perf_counter() - debut:.0f} s pour {len(resultat.item_results)} cas")
    return resultat


# Deux consignes de routage, comparées comme les deux versions de prompt de J4-03
routage_v1 = experience("helpdesk-routage", ev.tache_routage(llm, ev.CONSIGNE_ROUTAGE_V1), [ev.ev_routage, ev.ev_latence], "consigne v1 (J3)")
print("v1 :", ev.resume(routage_v1, MODELE))
routage = experience("helpdesk-routage", ev.tache_routage(llm, ev.CONSIGNE_ROUTAGE_V2), [ev.ev_routage, ev.ev_latence], "consigne v2")
print("v2 :", ev.resume(routage, MODELE))

In [ ]:
lignes = pd.DataFrame(ev.lignes_resultat(routage))
print("Erreurs de routage :")
print(lignes[lignes["routage_juste"] == 0][["cas", "specialiste", "refus", "origine"]].fillna("").to_string(index=False))

**👀 Ce que vous devez observer** (mesuré pendant la préparation, `granite4.1:8b`) :
- **consigne v1, celle du routeur de J3 : 20 %** de routage juste. Presque tout part vers `diagnostic`, et c'était pareil pour les trois modèles testés (exercice 4.1) : le défaut était dans la **consigne**, pas dans le modèle. J3-01 l'avait laissé entrevoir (« Outlook » → Diagnostic) ; sans mesure, ce routeur serait parti en production ;
- **consigne v2** (définition de chaque spécialiste et quatre exemples) : **85 %** (17 cas sur 20), avec une latence p95 plus basse (5,9 s contre 9,4 s) ;
- les échecs restants sont **instructifs** : C08 (« je voudrais une souris ») est **refusé par le garde-fou** de J1, faute de mot-clé (un faux positif) ; C19 (le poème sur les serveurs) **passe** le garde-fou (un faux négatif) ; C03 (« l'ERP est inaccessible ») part vers la KB, ce qui se défend (KB-008 est la procédure d'incident majeur). Quand un modèle « se trompe » de façon défendable, on **revoit l'annotation** : un dataset se corrige aussi.

Dans Langfuse, **Datasets → `helpdesk-ops-eval` → Runs** : le run, ses scores moyens, et pour chaque cas la trace de l'exécution.

## 4. L'appel d'outil et la qualification

In [ ]:
outils = experience("helpdesk-outils", ev.tache_outil(llm), [ev.ev_outil_valide, ev.ev_outil_attendu, ev.ev_latence])
print(ev.resume(outils, MODELE))
l = pd.DataFrame(ev.lignes_resultat(outils))
print(l[(l.get("outil_attendu", pd.Series(1, index=l.index)) == 0) | (l.get("outil_valide", pd.Series(1, index=l.index)) == 0)][["cas", "outils", "outil_valide", "outil_attendu"]].to_string(index=False))

In [ ]:
qualification = experience("helpdesk-qualification", ev.tache_qualification(llm), [ev.ev_qualification, ev.ev_latence])
print(ev.resume(qualification, MODELE))
l = pd.DataFrame(ev.lignes_resultat(qualification))
print(l[(l.get("categorie_juste", pd.Series(1, index=l.index)) == 0) | (l.get("priorite_juste", pd.Series(1, index=l.index)) == 0)]
        [["cas", "categorie", "categorie_juste", "priorite", "priorite_juste", "origine"]].fillna("").to_string(index=False))

**👀 Ce que vous devez observer :**
- `outil_valide` (la **forme** : un appel bien formé, vers un outil existant) est en général bien plus haut que `outil_attendu` (le **bon** outil) : un modèle peut appeler proprement le mauvais outil ;
- la qualification : mesuré en préparation, **31 %** de catégories justes et 69 % de priorités justes. Ce prompt est celui du jalon J2, **sans définition des catégories** : exactement le défaut de la v1 de J4-03 (50 % → 100 % avec les définitions). Prochaine étape évidente : versionner le prompt de qualification dans Langfuse et y reporter les définitions, puis relancer ce run.

> 💡 **Le « Tips » du Formateur :** un taux sans **intervalle** se sur-interprète. Sur 20 cas, un écart de 1 cas vaut 5 points : n'annoncez pas « +5 % » comme une victoire. Augmentez le dataset avant de trancher entre deux modèles proches.

## 5. 🧪 Exercice 4.1 (45 min)

➡️ `exercices/enonces/Ex4_1_comparer_les_modeles.ipynb` : lancer les mêmes experiments pour `granite4.1:3b`, `granite4.1:8b` et `gemma4:12b`, produire le tableau comparatif et **recommander** un modèle, chiffres à l'appui.

## 6. Le tableau de bord d'un responsable Ops

### [Théorie] Six questions, six widgets

Langfuse propose des **tableaux de bord personnalisés** (menu *Dashboards* : widgets construits sur les mêmes vues que l'API metrics). Avant de cliquer, on décide **quoi** montrer :

| Question | Widget | Seuil d'alerte proposé |
|---|---|---|
| Rapide ? | latence p95 par trace | p95 > 30 s en salle GPU |
| Fiable ? | observations ERROR par jour | hausse brutale d'un jour à l'autre |
| Juste ? | moyenne des scores `fidelite`, `routage_juste` | baisse après un changement de prompt |
| Apprécié ? | part de `pouce = false` | > 10 % |
| Coûteux ? | appels et tokens par modèle | dérive du volume par demande |
| Utilisé ? | sessions par jour | — |

Les mêmes chiffres, par code :

In [ ]:
from helpdesk_ops import tableau_bord as tb

print("Scores moyens (7 derniers jours)")
print(pd.DataFrame(tb.scores_moyens(langfuse)).rename(columns={"avg_value": "moyenne", "count_count": "nb"}).to_string(index=False))
print("\nLatence par trace (s)")
lat = pd.DataFrame(tb.latence_par_trace(langfuse))
lat[["p50_latency", "p95_latency"]] = (lat[["p50_latency", "p95_latency"]] / 1000).round(1)
print(lat.sort_values("p95_latency", ascending=False).head(8).to_string(index=False))

**👀 Ce que vous devez observer :** les scores des experiments du jour (`routage_juste`, `outil_valide`, `categorie_juste`…) et ceux de J4-04 (`pouce`, `fidelite`) sont maintenant interrogeables comme n'importe quel indicateur. Créez dans l'interface un tableau de bord *HelpDesk-Ops* avec au moins trois des widgets ci-dessus.

## 7. Ce qu'on retient du jour 4

| Brique | La phrase à retenir |
|---|---|
| Observabilité | un bug d'agent répond en HTTP 200 : on regarde la trace, les percentiles, les scores |
| Instrumentation | des observations typées, des niveaux, et rien de sensible qui ne soit nécessaire |
| RGPD | les traces s'effacent et se purgent par l'API : une procédure, une durée |
| Prompts | versionnés, liés aux traces, promus après mesure, avec un repli |
| Scores et juge | le juge est un instrument : juge ≠ évalué, calibré, éprouvé contre l'injection |
| Experiments | un dataset versionné, un run par variante, des taux et des cas ratés, `max_concurrency=1` en local |

### ✅ Quiz du jour

1. Pourquoi le dataset vit-il dans Git plutôt que seulement dans Langfuse ?
2. Pourquoi `max_concurrency=1` contre un Ollama local ?
3. `outil_valide` vaut 1,0 et `outil_attendu` 0,6. Qu'est-ce que cela dit du modèle ?
4. Comment effacer les traces d'une personne dans Langfuse open source ?
5. Quelles conditions avant de faire confiance à un LLM-as-judge ?
6. Un prompt v2 gagne 1 cas sur 20 : faut-il le promouvoir ?

<details>
<summary>Réponses</summary>

1. Il se relit, se versionne et évolue avec le code ; Langfuse en est une copie synchronisée (identifiants stables).
2. Sinon les cas s'exécutent en parallèle, font la queue dans Ollama et les latences mesurées sont fausses.
3. Il appelle toujours les outils proprement, mais pas toujours le bon : un problème de choix, pas de format.
4. Par l'API : `purger_traces(langfuse, user_id=… ou session_id=…, simulation=False)` ; la rétention automatique est une fonction Enterprise.
5. Juge ≠ modèle évalué, sortie structurée, calibration contre des cas annotés (dont des cas graves), tests de biais et d'injection.
6. Pas sur cette seule base : 1 cas = 5 points sur 20 ; agrandir le dataset, vérifier qu'aucun score clé ne régresse, puis décider (en J5 : par la CI).
</details>

**Demain, jour 5 :** production, sécurité et conformité. Les taux d'aujourd'hui deviennent une **gate de CI** ; le poème accepté et la source contredite deviennent des cas de **red-team**.